# This is sample project which perform a trend analysis on twitter topics. 
## Part 1 - Compare last 7 days trend on input topics
## Part 2 - Compare trend with the most trending topic

P.S - This notebook is built and executed on Azure Synapse Notebook

## Set Spark Configuration

In [1]:
%%configure
{
    
    "driverMemory": 
    {
        "activityParameterName" : "SparkPoolMemory",
        "defaultValue" : "28g",
    },
    "driverCores":
    {
        "activityParameterName" : "SparkPoolCores",
        "defaultValue" : 4,
    } ,
    "executorMemory": 
    {
        "activityParameterName" : "SparkPoolMemory",
        "defaultValue" : "28g"
    }, 
    "executorCores": {
        "activityParameterName" : "SparkPoolCores",
        "defaultValue" : 4
    },
    "conf":
    {
        "spark.sql.adaptive.enabled" : "True"
    }
}

StatementMeta(SparkPoolSmall, 0, -1, Finished, Available, Finished)

See https://go.microsoft.com/fwlink/?linkid=2170827

## Parameters

In [ ]:
topics = ["laptop","echt"]
inputTopictoCompareWithTrendingTopic = "laptop"
last7daysFromDate = "2012-02-05"
rawFilePath = "abfss://raw@<storageAccountName>.dfs.core.windows.net/twitter-sample.json"
outputPath = "abfss://processed@<storageAccountName>.dfs.core.windows.net/output.txt"
outputPathComparisonResult = "abfss://processed@<storageAccountName>.dfs.core.windows.net/comparisonOutput.txt"
Debug = True
NoOfRecordsTobeDisplayed = 2

StatementMeta(SparkPoolSmall, 0, 2, Finished, Available, Finished)

## Import Libraries and Set Spark Config

In [3]:
import pyspark.sql.functions as f
from pyspark.sql import Window
from pyspark.sql import DataFrame
from pyspark.sql.types import DateType, DoubleType
from dateutil import parser
from datetime import *
import numpy as np

#this can be set effectively for large datasets
#spark.conf.set("spark.sql.shuffle.partitions", 32)


StatementMeta(SparkPoolSmall, 0, 3, Finished, Available, Finished)

## DateUtility class

In [4]:
class DateUtility:

    def standardiseDates(self, date_text):
        """ 
        This method will standardise date values to yyyy-mm-dd irrespective of input format 

        Parameters
        ----------
        date_text : date value  

        Returns
        ---------
        date in yyyy-mm-dd format
        """
        try:
            if date_text is None:
                return None
            if(bool(parser.parse(date_text))):
                    extractedDate = parser.parse(date_text)
                    year = extractedDate.year
                    month = extractedDate.month
                    day = extractedDate.day
                    strDate =f"{year}-{month}-{day}"
                    return datetime.strptime(strDate, '%Y-%m-%d').date()
        except ValueError:
            return None



StatementMeta(SparkPoolSmall, 0, 4, Finished, Available, Finished)

## Data Preparation class

In [5]:
#This class conatins the functions which are used to prepare the data for trnd analysis 
class DataPreparation:

    def filterData(self, df:DataFrame, numberOfDays = 0,  lastdaysFromDate = "", arrTopics =[]):   
        
        """ 
        This method filters data based on dates and based on provided topics 

        Parameters
        ----------
        df : Dataframe contaning tweets  

        Returns
        ---------
        df : Dataframe with tweet data

        """   
        if numberOfDays != 0:
            df = df.filter(f.datediff(f.col("createdDate"), f.lit(last7daysFromDate))<7)

        if len(arrTopics) > 0 :
            df = df.filter(df["tweet"].isin(arrTopics))

        return df

    
    def prepareData(self, df:DataFrame):

        """ 
        This method extract words from all tweets, removes anything but alphabets, converts all tweets to lower case 

        Parameters
        ----------
        df : Dataframe contaning tweets, it should have two columns tweet and createdDate    

        Returns
        ---------
        df_Data : Dataframe with tweet data
        """

        #Prepare words array for each tweet
        df_Data = df.select(f.split(df["tweet"], " ").alias("arr_content"), "createdDate")

        #Explode the words array and convert every word to lower case
        df_Data = df_Data.select(f.explode(f.col("arr_content"))\
            .alias("topic"),"createdDate")\
            .withColumn("lower_word",f.lower(f.col("topic")))\
            .select("lower_word", "createdDate")

        #Cleanse the words by keeping only alphabets
        df_Data = df_Data.select(f.regexp_extract(f.col("lower_word"), "[a-z]+", 0)\
            .alias("tweet"),"createdDate")\
            .filter(f.col("tweet") != "")

        #Calculate the frequency of each word everyday
        df_Data = df_Data.groupBy("tweet", "createdDate").count()

        return df_Data
    

    def fillMissingDates(self, df: DataFrame):

        """ 
        This method adds missing dates with tweet count 0 to the dataset. Missing dates are added daily based on 
        minimum and maximum date in the dataset.

        Parameters
        ----------
        df : Dataframe contaning tweets along with the count of each tweet everyday, it should have two columns tweet, createdDate  and count  

        Returns
        ---------
        df_FrequncyWithAllDates : Dataframe with tweet data will all dates withing date range and occurance count
        """

        #Fill missing dates, 
        df_ContentWithAllDates = df.groupBy("tweet").agg(
            f.date_trunc("dd",f.max(f.to_date("createdDate"))).alias("max_date"),
            f.date_trunc("dd",f.min(f.to_date("createdDate"))).alias("min_date")
        ).select(
            "tweet", f.expr("sequence(min_date, max_date, interval 1 day)").alias("allDates")
        ).withColumn(
            "continuousCreatedDate", f.explode("allDates")).select(
                "tweet", "continuousCreatedDate"
        )

        df_FrequncyWithAllDates = df_ContentWithAllDates.join(df
            ,(df_ContentWithAllDates["tweet"] == df["tweet"]) & 
            (df_ContentWithAllDates["continuousCreatedDate"] == df["createdDate"])
            , "left").select(
                df_ContentWithAllDates["tweet"], 
                df_ContentWithAllDates["continuousCreatedDate"].alias("createdDate"),
                df["count"]
            )

        #Add 0 counts to the missing dates
        df_FrequncyWithAllDates = df_FrequncyWithAllDates.withColumn("count", f.when(
            df_FrequncyWithAllDates["count"].isNull(),0
            ).otherwise(
                df_FrequncyWithAllDates["count"]
                )
        )

        return df_FrequncyWithAllDates


    
    def buildXYColumnsToCalculateTrendSlope(self, df:DataFrame):

        """ 
        This method add date ranks and builds x (date ranks) and y(count of ocuurance) arrays to be used to calculate slope 

        Parameters
        ----------
        df : Dataframe contaning tweets, it should have two columns tweet , createdDate  and count  

        Returns
        ---------
        df : Dataframe with tweet column as text, array columns countList and dateRankList
        """

        #Add Date ranks
        df = df.withColumn("dateRank", f.dense_rank().over(
            Window.partitionBy("tweet").orderBy("createdDate")
            )
        )

        #Create count arrays and daterank arrays to calculate slope
        df = df.groupBy("tweet").agg(
            f.collect_list("count").alias("countList"),
            f.collect_list("dateRank").alias("dateRankList")
        )
        return df

StatementMeta(SparkPoolSmall, 0, 5, Finished, Available, Finished)

## Trend class

In [6]:
#This class contains functions related to Trend Analysis
class Trend:


    def calculateTrendSlope(self, x,y, order = 1):
        """ 
        This function returns the slope after polynomial fit of the input points 

        Parameters
        ----------
        x : Array of numbers
        y : Array of numbers
        order  : order of polynomial fit 

        Returns
        ---------
        slope : slope between 2 points
        """
        coeffs = np.polyfit(x,y,order)
        slope = coeffs[-2]
        return float(slope)

    
    def compareWithMostTrendingTopic(self, df :DataFrame, inputTopic):
        """ 
        This method compares the slope of occurance with the maximum slope of occurance in the dataset 

        Parameters
        ----------
        df : Dataframe contaning tweetsand their trend slope  

        Returns
        ---------
        df : Dataframe with tweet, slope and slope diffrence with maximum slope
        """
        df = df.withColumn("maxSlope", f.max("slope").over(Window.partitionBy(f.lit(1))))\
            .withColumn("slopeDifference", f.col("maxSlope") - f.col("slope"))
        df = df.filter(f.col("tweet") == inputTopic).select("tweet","slope","slopeDifference")
        return df



StatementMeta(SparkPoolSmall, 0, 6, Finished, Available, Finished)

In [7]:
#Create object for DateUtility, DataPreparations and Trend class
dateUtility = DateUtility()
dataPreparation = DataPreparation()
trend = Trend()

#Convert Python functions to be applied on dataframe column
standardiseDates_udf = f.udf(dateUtility.standardiseDates, DateType())
calculateSlope_udf = f.udf(trend.calculateTrendSlope, returnType=DoubleType())

StatementMeta(SparkPoolSmall, 0, 7, Finished, Available, Finished)

## Part 1
Compare last 7 days trends of two input topics

### Read raw data

In [8]:
#Read the file
df_TwitterSample = spark.read.load(rawFilePath, format='json')


StatementMeta(SparkPoolSmall, 0, 8, Finished, Available, Finished)

In [10]:
df_TwitterSample.printSchema()


StatementMeta(SparkPoolSmall, 0, 10, Finished, Available, Finished)

root
 |-- _corrupt_record: string (nullable = true)
 |-- demographic: struct (nullable = true)
 |    |-- gender: string (nullable = true)
 |-- facebook: struct (nullable = true)
 |    |-- application: string (nullable = true)
 |    |-- author: struct (nullable = true)
 |    |    |-- avatar: string (nullable = true)
 |    |    |-- id: string (nullable = true)
 |    |    |-- link: string (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |-- caption: string (nullable = true)
 |    |-- description: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- likes: struct (nullable = true)
 |    |    |-- count: long (nullable = true)
 |    |    |-- ids: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |    |-- names: array (nullable = true)
 |    |    |    |-- element: string (containsNull = true)
 |    |-- link: string (nullable = true)
 |    |-- message: string (nullable = true)
 |    |-- name: string (nullable = tru

### Prepare the data for trend analysis

In [15]:
#Extract the date and the contents of the tweet
df_Content = df_TwitterSample.withColumn("tweet",f.col("twitter.text")).withColumn("created_at",f.col("twitter.created_at")).select("tweet","created_at")

#Standarize created_at
df_Content = df_Content.withColumn("createdDate", standardiseDates_udf('created_at')).select("tweet", "createdDate")

#Filter the date based on dates
df_Content = dataPreparation.filterData(df_Content, numberOfDays= 7,  lastdaysFromDate= last7daysFromDate)

#Preapare the data
df_Content = dataPreparation.prepareData(df_Content)

#The dataset has only two days of data , and the fill missing dates add the missing date based on the date range present in the data set.
#real world data set would not definitely have 2 days of data and ask for last 7 days of analysis. A method can easily be written 
#to backfill missing dates using number of days , however I thought this approach is much more dynamic in nature. 
df_Content = dataPreparation.fillMissingDates(df_Content)

#Builds the x, y data in array format for calculating the slope
df_Content = dataPreparation.buildXYColumnsToCalculateTrendSlope(df_Content)

#filters data based on 2 input topics, this filter can be called in earlier stage as well to minimize the data to be processed
df_ContentFiltered = dataPreparation.filterData(df_Content,arrTopics=topics)


if Debug == True:
    display(df_Content.limit(NoOfRecordsTobeDisplayed))
    display(df_ContentFiltered.limit(NoOfRecordsTobeDisplayed))

StatementMeta(SparkPoolSmall, 6, 13, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, fa056c98-e85d-4456-8d93-f57cf0d13192)

### Calculate Slope

In [17]:
df_Slopes = df_ContentFiltered.withColumn(
    "slope", calculateSlope_udf(df_ContentFiltered["countList"], df_ContentFiltered["dateRankList"])
)

if Debug == True:
    display(df_Slopes.limit(NoOfRecordsTobeDisplayed))

StatementMeta(SparkPoolSmall, 6, 15, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 0767964b-04db-4440-a7e8-4fbd131780a4)

### Write the final output

In [18]:
df_Output = df_Slopes.withColumn(
    "output", f.concat(f.col("tweet"), f.lit(" : "), f.col("slope"))
    ).select(
"output"
)

#Write the data
df_Output.write.format("text").option("header", "false").mode("overwrite").save(outputPath)

StatementMeta(SparkPoolSmall, 6, 16, Finished, Available, Finished)

## Part 2
### If the user provides one topic as input the program chooses the most trending topic from dataset and compares the trends with provided topic.

### Calculate slope for all tweets

In [41]:
df_AllSlopes = df_Content.withColumn(
    "slope", calculateSlope_udf(df_Content["countList"], df_Content["dateRankList"])
)

if Debug == True:
    display(df_AllSlopes.limit(NoOfRecordsTobeDisplayed))

StatementMeta(SparkPoolSmall, 6, 39, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, a4395d3d-c965-498a-acb6-79b973999cf1)

### Compare slope of the input topic with max slope

In [48]:
df_Result = trend.compareWithMostTrendingTopic(df_AllSlopes, inputTopictoCompareWithTrendingTopic)

if Debug == True:
    display(df_Result.limit(1))

StatementMeta(SparkPoolSmall, 6, 46, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 5f801718-81eb-4b3e-b8af-47f00e5258e7)

### Write the comparison result

In [50]:
df_Result.withColumn(
    "output", f.concat(f.col("tweet"), f.lit(" slope : "), f.col("slope"), f.lit(", slopeDifference : "), f.col("slopeDifference"), )
    ).select(
"output"
).write.format("text").option("header", "false").mode("overwrite").save(outputPathComparisonResult)

StatementMeta(SparkPoolSmall, 6, 48, Finished, Available, Finished)